# generate_board

Compose an ArUco board set - QR metadata plus a slot grid on a background preset -
and save the on-screen PNGs and the two config JSONs an ingest run needs.

- **Does:** builds `TOTAL_SHEETS` board images with `BoardImageComposer`, derives
  the background mask into the ingest config, and writes everything under
  `data/aruco_boards/{BOARD_CONFIG_ID}/`.
- **Needs:** nothing but `src` and the parameters below; no input data.
- **Produces:** `sheet_XX.png` (one per sheet), `{id}_ArucoBoardConfig.json`, and
  `{id}_SheetArucoConfig.json` (mask already enabled for green/blue presets).
- **Recreate the data:** run this notebook top to bottom. The output folder is
  gitignored; nothing is committed.

Boards are displayed on a screen and photographed; the QR `sheet_index` plus the
slot label (e.g. `B3`) are the manual key that ties a photo back to a board.

Why the preset and mask matter: see
[docs/guides/green_background.md](../docs/guides/green_background.md). This
notebook only drives the pieces; it defines no logic of its own.


In [ ]:
import json

import cv2
import matplotlib.pyplot as plt

from snap_fit.aruco.board_config_resolver import derive_background_mask
from snap_fit.aruco.board_image_composer import BoardImageComposer
from snap_fit.aruco.sheet_metadata import SheetMetadata
from snap_fit.aruco.sheet_metadata import SheetMetadataDecoder
from snap_fit.config.aruco.aruco_board_config import ArucoBoardConfig
from snap_fit.config.aruco.aruco_detector_config import ArucoDetectorConfig
from snap_fit.config.aruco.metadata_zone_config import MetadataZoneConfig
from snap_fit.config.aruco.metadata_zone_config import SlotGridConfig
from snap_fit.config.aruco.sheet_aruco_config import SheetArucoConfig
from snap_fit.params.snap_fit_params import get_snap_fit_params

In [ ]:
# --- parameters -----------------------------------------------------------
# Board identity. The id names the output folder and is embedded in every QR.
TAG = "gd_s"
BOARD_CONFIG_ID = "greendemo_small"
TOTAL_SHEETS = 4

# Background preset. green/blue auto-enable the ingest mask; white leaves it off.
BACKGROUND_PRESET = "green"
# BACKGROUND_PRESET = "white"  # white boards stay valid, mask-free
# BACKGROUND_PRESET = "blue"

# Slot grid drawn on each board (labels A1.. across rows x cols).
GRID_ROWS = 2
GRID_COLS = 2

# Redundant QR copies per board (more copies survive a worse photo).
QR_N_CODES = 3

# Minimum contour area (px^2) for piece detection at THIS board's scale.
# greendemo pieces measure ~10k-16k px^2 rectified, far below the 80k global
# default, so this value travels with the saved ingest config.
MIN_AREA = 5_000

In [ ]:
params = get_snap_fit_params()
save_dir = params.paths.aruco_board_fol / BOARD_CONFIG_ID

board_config = ArucoBoardConfig(
    background_preset=BACKGROUND_PRESET,
    markers_x=4,
    markers_y=5,
    marker_length=100,
    marker_separation=40,
)
metadata_zone = MetadataZoneConfig(
    enabled=True,
    qr_n_codes=QR_N_CODES,
    qr_ecc="M",
    text_enabled=True,
    slot_grid=SlotGridConfig(rows=GRID_ROWS, cols=GRID_COLS),
)
sheet_aruco_config = SheetArucoConfig(
    min_area=MIN_AREA,
    detector=ArucoDetectorConfig(board=board_config),
    metadata_zone=metadata_zone,
)
# green/blue enable the background mask; white leaves it disabled (same call, D14).
sheet_aruco_config = derive_background_mask(sheet_aruco_config)
sheet_aruco_config.preprocess.background_mask

In [ ]:
save_dir.mkdir(parents=True, exist_ok=True)
composer = BoardImageComposer(board_config, metadata_zone)

board_images = []
for i in range(TOTAL_SHEETS):
    metadata = SheetMetadata(
        tag_name=TAG,
        sheet_index=i,
        total_sheets=TOTAL_SHEETS,
        board_config_id=BOARD_CONFIG_ID,
    )
    img = composer.compose(metadata)
    out_path = save_dir / f"sheet_{i:02d}.png"
    cv2.imwrite(str(out_path), img)
    board_images.append(img)
    print(f"saved {out_path.name}  ({img.shape[1]}x{img.shape[0]} px)")

board_cfg_path = save_dir / f"{BOARD_CONFIG_ID}_ArucoBoardConfig.json"
board_cfg_path.write_text(json.dumps(board_config.model_dump(), indent=2))
sheet_cfg_path = save_dir / f"{BOARD_CONFIG_ID}_SheetArucoConfig.json"
sheet_cfg_path.write_text(sheet_aruco_config.model_dump_json(indent=2))
print(f"saved configs to {save_dir}")

In [ ]:
# Show the composed board(s) inline for visual validation.
fig, axes = plt.subplots(1, len(board_images), figsize=(6 * len(board_images), 8))
axes = axes if len(board_images) > 1 else [axes]
for i, img in enumerate(board_images):
    axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"sheet_{i:02d}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Validate: every saved board's QR round-trips to the expected board id.
decoder = SheetMetadataDecoder()
for i in range(TOTAL_SHEETS):
    path = save_dir / f"sheet_{i:02d}.png"
    meta = decoder.decode(cv2.imread(str(path)))
    if meta is None:
        msg = f"QR did not decode on {path.name}"
        raise RuntimeError(msg)
    if meta.board_config_id != BOARD_CONFIG_ID:
        msg = f"unexpected board id {meta.board_config_id!r} on {path.name}"
        raise RuntimeError(msg)
    print(f"QR OK on {path.name}: board_config_id={meta.board_config_id}")

In [ ]:
# Validate: the saved ingest config carries the mask for green/blue presets.
reloaded = SheetArucoConfig.model_validate_json(sheet_cfg_path.read_text())
mask = reloaded.preprocess.background_mask
if BACKGROUND_PRESET in {"green", "blue"}:
    if mask is None or not mask.enabled:
        msg = "saved SheetArucoConfig should have the background mask enabled"
        raise RuntimeError(msg)
    print(f"mask enabled: mode={mask.mode}")
else:
    print(f"no background mask for preset {BACKGROUND_PRESET!r} (expected)")
print("board set ready.")